# The Transformer

Companion code for the *Predictive Analytics Lab* (section 23). Run the cell below.

In [ ]:
# The lab's toy Transformer: translate "the cat is tired" into Spanish, step by step
import numpy as np

D = 8; SD = np.sqrt(D)          # dims: pos-a, pos-b, noun, animal, verb, adj, det, end
EN = {"the": [0,0,0,0,0,0,1,0], "cat": [0,0,1,1,0,0,0,0], "is": [0,0,0,0,1,0,0,0], "tired": [0,0,0,0,0,1,0,0], "<eos>": [0,0,0,0,0,0,0,1]}
ES = {"<s>": [0,0,0,0,0,0,0,1], "el": [0,0,0,0,0,0,1,0], "gato": [0,0,1,1,0,0,0,0], "casa": [0,0,1,0,0,0,0,0],
      "está": [0,0,0,0,1,0,0,0], "cansado": [0,0,0,0,0,1,0,0], "</s>": [0,0,0,0,0,0,0,1]}
OUT = ["el", "gato", "casa", "está", "cansado", "</s>"]

def pe(p): return np.array([np.sin(p / 10000 ** (i / D)) if i % 2 == 0 else np.cos(p / 10000 ** ((i - 1) / D)) for i in range(D)])
def embed(toks, table): return np.array([np.array(table[t]) * SD + pe(p) for p, t in enumerate(toks)])
def W(entries, r=8, c=4):
    M = np.zeros((r, c))
    for i, j, v in entries: M[i, j] = v
    return M
def softmax(E): E = E - E.max(axis=1, keepdims=True); A = np.exp(E); return A / A.sum(axis=1, keepdims=True)
def attn(X, Wq, Wk, Wv, dk, mask=False, Xkv=None):
    Xkv = X if Xkv is None else Xkv
    Q, K, V = X @ Wq, Xkv @ Wk, Xkv @ Wv
    E = Q @ K.T / np.sqrt(dk)
    if mask: E = np.where(np.triu(np.ones_like(E), 1) == 1, -np.inf, E)
    A = softmax(E); return A @ V, A
def layernorm(X): return (X - X.mean(1, keepdims=True)) / np.sqrt(X.var(1, keepdims=True) + 1e-5)

# Encoder weights (hand-set): head 1 = meaning, head 2 = previous word
c1, s1, g = np.cos(1), np.sin(1), 3
WQ1 = W([(5,0,.5), (4,0,.5), (6,0,.5), (2,1,.5)]); WK1 = W([(2,0,.5), (3,0,.5), (5,1,.5)]); WV1 = W([(2,0,.5), (3,1,.5), (4,2,.5), (5,3,.5)])
WQ2 = W([(0,0,g*c1), (1,0,-g*s1), (1,1,g*c1), (0,1,g*s1)]); WK2 = W([(0,0,1), (1,1,1)]); WV2 = W([(2,0,.5), (4,1,.5), (5,2,.5), (6,3,.5)])
WO = np.zeros((8, 8))
for a, b in [(0,2), (1,3), (2,4), (3,5)]: WO[a, b] = .6
for a, b in [(4,2), (5,4), (6,5), (7,6)]: WO[a, b] = .3
seed = 7
def rr():
    global seed; seed = (seed * 16807) % 2147483647; return seed / 2147483647 - .5
W1 = np.array([[rr() * .4 for _ in range(16)] for _ in range(8)]); W2 = np.array([[rr() * .4 for _ in range(8)] for _ in range(16)])

# ---- Encoder ----
src = ["the", "cat", "is", "tired", "<eos>"]
X = embed(src, EN)
Z1, A1 = attn(X, WQ1, WK1, WV1, 4); Z2, A2 = attn(X, WQ2, WK2, WV2, 4)
X1 = layernorm(X + np.c_[Z1, Z2] @ WO)                       # multi-head + Add & Norm
ENC = layernorm(X1 + np.maximum(0, X1 @ W1) @ W2)           # feed-forward + Add & Norm
print("Head 1 attention (rows = query word):\n", A1.round(2))
print(f'"tired" attends to "cat" with weight {A1[3, 1]:.2f}\n')

# ---- Decoder ----
WQs = W([(0,0,g), (1,1,g)]); WKs = W([(0,0,1), (1,1,1)]); WVs = W([(2,0,.5), (3,1,.5), (4,2,.5), (5,3,.5)])
WOs = np.zeros((4, 8))
for a, b in [(0,2), (1,3), (2,4), (3,5)]: WOs[a, b] = .3
WQc = np.zeros((8, 8))
for a, b in [(7,6), (6,2), (2,4), (4,5), (5,7)]: WQc[a, b] = np.sqrt(2)   # "what should come next?"
WKc = np.diag([0, 0, 1, 1, 1, 1, 1, 1.]); WVc = WKc.copy(); WOc = 12 * WKc

def decode(prefix):
    Y = embed(prefix, ES)
    Zs, _ = attn(Y, WQs, WKs, WVs, 4, mask=True); Y1 = layernorm(Y + Zs @ WOs)     # masked self-attention
    Zc, Ac = attn(Y1, WQc, WKc, WVc, 8, Xkv=ENC); Y2 = layernorm(Y1 + Zc @ WOc)    # encoder-decoder attention
    logits = np.array([np.dot(np.array(ES[w]) * SD, Y2[-1]) for w in OUT])          # linear (tied weights)
    p = np.exp(logits - logits.max()); p /= p.sum()                                 # softmax
    return OUT[p.argmax()], p, Ac[-1]

prefix = ["<s>"]
while True:
    word, p, a = decode(prefix)
    print(f"{' '.join(prefix):22s} -> {word:8s} p={p.max():.2f}   looks at '{src[a.argmax()]}' ({a.max():.0%})")
    if word == "</s>" or len(prefix) > 6: break
    prefix.append(word)
print("\nTranslation:", " ".join(prefix[1:]))